# Silhouette Coefficient Code Companion

This notebook connects Silhouette Coefficient code with the theory: intra-cluster distance, inter-cluster distance, score range, and choosing K.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples


## Create Cluster Data


In [ ]:
X, _ = make_blobs(
    n_samples=300,
    centers=4,
    cluster_std=0.8,
    random_state=42
)

points = pd.DataFrame(X, columns=["feature_1", "feature_2"])
points.head()


## Compare Silhouette Scores for Different K Values

A higher average silhouette score usually means clusters are better separated and points fit their assigned clusters well.


In [ ]:
rows = []
for k in range(2, 9):
    model = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = model.fit_predict(points)
    rows.append({
        "k": k,
        "silhouette_score": silhouette_score(points, labels),
        "inertia": model.inertia_
    })

results = pd.DataFrame(rows)
results


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(results["k"], results["silhouette_score"], marker="o")
plt.xlabel("K")
plt.ylabel("Average silhouette score")
plt.title("Silhouette Score for Different K Values")
plt.tight_layout()
plt.show()


## Inspect Individual Silhouette Values

Individual silhouette values show how well each point fits its own cluster.


In [ ]:
best_k = int(results.sort_values("silhouette_score", ascending=False).iloc[0]["k"])
model = KMeans(n_clusters=best_k, n_init=10, random_state=42)
labels = model.fit_predict(points)

sample_scores = silhouette_samples(points, labels)

silhouette_table = points.copy()
silhouette_table["cluster"] = labels
silhouette_table["silhouette_score"] = sample_scores
silhouette_table.head(10)


In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(points["feature_1"], points["feature_2"], c=sample_scores, cmap="coolwarm", alpha=0.8)
plt.colorbar(label="Silhouette score")
plt.xlabel("feature_1")
plt.ylabel("feature_2")
plt.title(f"Point-Level Silhouette Scores for K={best_k}")
plt.tight_layout()
plt.show()
